# Keypoint Corrections

1. Smoothen foot/ankle keypoints
2. Correct hip keypoints

## 1. Smooth foot/ankle keypoints

Using Savitzky–Golay filter

In [1]:
import os
import matplotlib
matplotlib.use('Agg')
from pathlib import Path
import pandas as pd
from scipy.signal import savgol_filter
import matplotlib.pyplot as plt
import glob
import numpy as np

UP_DIR = Path('../data/processed/keypoints_combined')
DOWN_DIR = Path('../data/processed/keypoints_combined_down')

SAVGOY_LANDMARKS = [
  'left_ankle_x', 'left_ankle_y', 'right_ankle_x', 'right_ankle_y',
  'left_heel_x', 'left_heel_y', 'right_heel_x', 'right_heel_y',
  'left_foot_index_x', 'left_foot_index_y', 'right_foot_index_x', 'right_foot_index_y'
]

UP_TIME_DICT = {
  '028': (9.90, 10.44, 10.60, 11.20, 11.20, 11.77),
  '030': (5.80, 6.12, 6.28,  6.80,  6.80,  7.63),
  '045': (6.54, 6.74, 6.90,  7.36,  7.36,  7.98),
  '047': (7.03, 7.36, 7.52,  7.88,  7.88,  8.80),
  '059': (6.65, 6.90, 7.06,  8.10,  8.10,  9.11),
  '061': (7.00, 7.40, 7.56,  8.20,  8.20,  9.00),
  '074': (6.07, 6.22, 6.38,  6.80,  6.80,  7.27),
  '076': (5.20, 5.80, 5.96,  6.34,  6.34,  7.20),
  '091': (6.39, 6.52, 6.68,  7.14,  7.14,  7.61),
  '093': (5.86, 6.22, 6.38,  6.74,  6.74,  7.50),
}

DOWN_TIME_DICT = {
  '028': (11.87, 11.90, 12.06, 12.57, 12.82),
  '030': (8.07, 9.70, 9.94, 10.64, 11.16),
  '045': (7.94, 7.96, 8.04, 8.74, 9.26),
  '047': (9.20, 10.14, 10.36, 11.10, 11.70),
  '059': (9.11, 9.18, 9.50, 10.37, 11.06),
  '061': (10.57, 10.88, 11.26, 12.26, 13.00),
  '074': (7.27, 7.30, 7.42, 7.88, 8.14),
  '076': (8.27, 8.72, 9.08, 9.73, 10.32),
  '091': (7.56, 7.58, 7.70, 8.09, 8.18),
  '093': (8.50, 9.24, 9.56, 10.08, 10.74),
}

WINDOW_SIZE = 11
POLY_DEGREE = 3

In [2]:
sagvoy_dict = {}

for dir in [(UP_DIR, 'up'), (DOWN_DIR, 'down')]:
    out_dir = Path(str(dir[0]).replace('combined', 'corrected'))
    out_dir.mkdir(parents=True, exist_ok=True)
    direction_dict = {}
    for file in dir[0].iterdir():
        data = pd.read_csv(file)
        time_data = data['time']
        foot_data = data[SAVGOY_LANDMARKS]
        smoothened_foot_data = savgol_filter(foot_data, WINDOW_SIZE, POLY_DEGREE, axis=0)
        smoothened_df = pd.DataFrame(smoothened_foot_data, columns=SAVGOY_LANDMARKS)

        data[smoothened_df.columns] = smoothened_df

        video_dict = {
            'time': time_data,
            'original': foot_data,
            'smooth': smoothened_df,
            'df': data.copy(),
            'out_path': out_dir / file.name,
        }
        direction_dict[file.stem] = video_dict
        data.to_csv(out_dir / file.name, index=None)
    sagvoy_dict[dir[1]] = direction_dict

In [3]:
landmark = 'left_ankle_y'
direction = 'down'
video = 'DJI_20250425092743_0028_D'

original_path = DOWN_DIR if direction == 'down' else UP_DIR
original_path = original_path / f'{video}.csv'
smooth_path = Path(str(original_path).replace('combined', 'corrected'))

original = pd.read_csv(original_path)
time = original['time'].values
original = original[landmark].values
smooth = pd.read_csv(smooth_path)[landmark].values

plt.plot(time, original, label='Original', color='blue')
plt.plot(time, smooth, label='Smooth', color='red')
plt.grid()
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.title(f"Example: {video.split('_')[2]} {landmark} ({direction})")
plt.show()

/var/folders/9k/2kqbhj4s4cnf1w60r_907q940000gn/T/ipykernel_62223/569392021.py:21: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 2. Hip corrections

Move hips and apply symmetry when needed

### 2.1. sit-to-stand

In [4]:
import re

HIP_CORRECTION_CONFIG = {
    '028': {'alpha_left': 0, 'alpha_right': 0, 'time_offset': 0},
    '030': {'alpha_left': -0.50, 'alpha_right': -0.50, 'time_offset': 0.40},
    '045': {'alpha_left': 0, 'alpha_right': 0, 'time_offset': 0},
    '047': {'alpha_left': -0.50, 'alpha_right': -0.38, 'time_offset': 0.32},
    '059': {'alpha_left': 0, 'alpha_right': 0, 'time_offset': 0},
    '061': {'alpha_left': -0.37, 'alpha_right': -0.34, 'time_offset': 0.38},
    '074': {'alpha_left': -0.40, 'alpha_right': -0.28, 'time_offset': 0.20},
    '076': {'alpha_left': -0.53, 'alpha_right': -0.42, 'time_offset': 0.35},
    '091': {'alpha_left': 0, 'alpha_right': 0, 'time_offset': 0},
    '093': {'alpha_left': -0.12, 'alpha_right': -0.08, 'time_offset': 0.27},
}

def project_to_length(hip, knee, length):
    vec = hip - knee
    norm = np.linalg.norm(vec)
    if norm < 1e-6:
        return hip
    return knee + vec / norm * length

def smooth_downward(y, start, end, dmax):
    y_new = y.copy()
    n = end - start + 1
    alpha = np.linspace(0, 1, n)
    s = alpha**2 * (3 - 2*alpha)
    offset = dmax * s
    offset = np.maximum.accumulate(offset)
    y_new[start:end+1] = y[start:end+1] + offset
    y_new[end+1:] = y[end+1:] + offset[-1]
    return y_new

def smooth_upward(y, start, end, dmax):
    """For sit-to-stand: large correction at deep sit (start), decreases to 0 at end."""
    y_new = y.copy()
    n = end - start + 1
    alpha = np.linspace(0, 1, n)
    s = alpha**2 * (3 - 2*alpha)
    offset = dmax * (1 - s)
    y_new[start:end+1] = y[start:end+1] + offset
    # After end (standing phase): no correction needed
    return y_new

def correct_hips_smooth(df, mask, alpha_left, alpha_right):
    if alpha_left == 0 and alpha_right == 0:
        return df
    df = df.copy()

    idx = np.where(mask.values)[0]
    if len(idx) == 0:
        return df
    start, end = 0, int(idx[-1])

    lh_x = df['left_hip_x'].values.copy()
    lh_y = df['left_hip_y'].values.copy()
    rh_x = df['right_hip_x'].values.copy()
    rh_y = df['right_hip_y'].values.copy()
    lk_x = df['left_knee_x'].values
    lk_y = df['left_knee_y'].values
    rk_x = df['right_knee_x'].values
    rk_y = df['right_knee_y'].values

    L_femur = np.sqrt((lh_x - lk_x)**2 + (lh_y - lk_y)**2)
    R_femur = np.sqrt((rh_x - rk_x)**2 + (rh_y - rk_y)**2)

    mean_L = np.nanmean(L_femur[idx])
    mean_R = np.nanmean(R_femur[idx])

    if alpha_left != 0:
        lh_y = smooth_upward(lh_y, start, end, abs(alpha_left) * mean_L)
    if alpha_right != 0:
        rh_y = smooth_upward(rh_y, start, end, abs(alpha_right) * mean_R)

    for i in range(len(df)):
        row = df.index[i]
        if alpha_left != 0:
            lh = project_to_length(np.array([lh_x[i], lh_y[i]]),
                                   np.array([lk_x[i], lk_y[i]]), L_femur[i])
            df.at[row, 'left_hip_x'], df.at[row, 'left_hip_y'] = lh[0], lh[1]
            lh_x[i], lh_y[i] = lh[0], lh[1]
        if alpha_right != 0:
            rh = project_to_length(np.array([rh_x[i], rh_y[i]]),
                                   np.array([rk_x[i], rk_y[i]]), R_femur[i])
            df.at[row, 'right_hip_x'], df.at[row, 'right_hip_y'] = rh[0], rh[1]
            rh_x[i], rh_y[i] = rh[0], rh[1]
    return df

def correct_hips_symmetry(df):
    """Average left/right for shoulder, hip, knee, ankle (x and y).
    Enforces bilateral symmetry in all joints used by angle formulas."""
    df = df.copy()
    for joint in ['shoulder', 'hip', 'knee', 'ankle']:
        for coord in ['x', 'y']:
            l = df[f'left_{joint}_{coord}'].values
            r = df[f'right_{joint}_{coord}'].values
            mean_val = (l + r) / 2
            df[f'left_{joint}_{coord}']  = mean_val
            df[f'right_{joint}_{coord}'] = mean_val
    return df

def get_sitting_mask(df, vid, time_offset=0):
    return df['time'] <= UP_TIME_DICT[vid][4] + time_offset

for stem, vdict in sagvoy_dict['up'].items():
    match = re.search(r'_0(\d{3})_D', stem)
    if not match:
        continue
    vid = match.group(1)
    if vid not in HIP_CORRECTION_CONFIG:
        continue

    cfg  = HIP_CORRECTION_CONFIG[vid]
    df   = vdict['df'].copy()
    mask = get_sitting_mask(df, vid, cfg['time_offset'])

    df = correct_hips_smooth(df, mask, cfg['alpha_left'], cfg['alpha_right'])
    df = correct_hips_symmetry(df)

    df.to_csv(vdict['out_path'], index=False)
    print(f"up {vid}: alpha_left={cfg['alpha_left']}, alpha_right={cfg['alpha_right']}, time_offset={cfg['time_offset']}, sitting frames={mask.sum()}")


up 030: alpha_left=-0.5, alpha_right=-0.5, time_offset=0.4, sitting frames=71
up 061: alpha_left=-0.37, alpha_right=-0.34, time_offset=0.38, sitting frames=130
up 076: alpha_left=-0.53, alpha_right=-0.42, time_offset=0.35, sitting frames=85
up 045: alpha_left=0, alpha_right=0, time_offset=0, sitting frames=42
up 091: alpha_left=0, alpha_right=0, time_offset=0, sitting frames=38


up 093: alpha_left=-0.12, alpha_right=-0.08, time_offset=0.27, sitting frames=58
up 074: alpha_left=-0.4, alpha_right=-0.28, time_offset=0.2, sitting frames=47
up 028: alpha_left=0, alpha_right=0, time_offset=0, sitting frames=66
up 047: alpha_left=-0.5, alpha_right=-0.38, time_offset=0.32, sitting frames=59
up 059: alpha_left=0, alpha_right=0, time_offset=0, sitting frames=73


### 2.2. stand-to-sit

In [5]:
HIP_CORRECTION_CONFIG_DOWN = {
    '028': {'alpha_left': -0.20, 'alpha_right': -0.12, 'time_offset': 0.50},
    '030': {'alpha_left': -0.40, 'alpha_right': -0.37, 'time_offset': 0.60},
    '045': {'alpha_left': -0.35, 'alpha_right': -0.28, 'time_offset': 0.70},
    '047': {'alpha_left': -0.58, 'alpha_right': -0.52, 'time_offset': 0.90},
    '059': {'alpha_left': -0.10, 'alpha_right':  0,    'time_offset': 0.50},
    '061': {'alpha_left': -0.30, 'alpha_right': -0.25, 'time_offset': 0.80},
    '074': {'alpha_left': -0.35, 'alpha_right': -0.30, 'time_offset': 0.50},
    '076': {'alpha_left': -0.50, 'alpha_right': -0.42, 'time_offset': 0.70},
    '091': {'alpha_left': -0.42, 'alpha_right': -0.38, 'time_offset': 0.30},
    '093': {'alpha_left': -0.12, 'alpha_right': -0.08, 'time_offset': 0.50},
}

def get_sitting_mask_down(df, vid, time_offset=0):
    return df['time'] >= DOWN_TIME_DICT[vid][3] - time_offset

def correct_hips_smooth_down(df, mask, alpha_left, alpha_right):
    if alpha_left == 0 and alpha_right == 0:
        return df
    df = df.copy()

    idx = np.where(mask.values)[0]
    if len(idx) == 0:
        return df
    start, end = int(idx[0]), len(df) - 1

    lh_x = df['left_hip_x'].values.copy()
    lh_y = df['left_hip_y'].values.copy()
    rh_x = df['right_hip_x'].values.copy()
    rh_y = df['right_hip_y'].values.copy()
    lk_x = df['left_knee_x'].values
    lk_y = df['left_knee_y'].values
    rk_x = df['right_knee_x'].values
    rk_y = df['right_knee_y'].values

    L_femur = np.sqrt((lh_x - lk_x)**2 + (lh_y - lk_y)**2)
    R_femur = np.sqrt((rh_x - rk_x)**2 + (rh_y - rk_y)**2)

    mean_L = np.nanmean(L_femur[idx])
    mean_R = np.nanmean(R_femur[idx])

    if alpha_left != 0:
        lh_y = smooth_downward(lh_y, start, end, abs(alpha_left) * mean_L)
    if alpha_right != 0:
        rh_y = smooth_downward(rh_y, start, end, abs(alpha_right) * mean_R)

    for i in range(len(df)):
        row = df.index[i]
        if alpha_left != 0:
            lh = project_to_length(np.array([lh_x[i], lh_y[i]]),
                                   np.array([lk_x[i], lk_y[i]]), L_femur[i])
            df.at[row, 'left_hip_x'], df.at[row, 'left_hip_y'] = lh[0], lh[1]
            lh_x[i], lh_y[i] = lh[0], lh[1]
        if alpha_right != 0:
            rh = project_to_length(np.array([rh_x[i], rh_y[i]]),
                                   np.array([rk_x[i], rk_y[i]]), R_femur[i])
            df.at[row, 'right_hip_x'], df.at[row, 'right_hip_y'] = rh[0], rh[1]
            rh_x[i], rh_y[i] = rh[0], rh[1]
    return df

for stem, vdict in sagvoy_dict['down'].items():
    match = re.search(r'_0(\d{3})_D', stem)
    if not match:
        continue
    vid = match.group(1)
    if vid not in HIP_CORRECTION_CONFIG_DOWN:
        continue

    cfg  = HIP_CORRECTION_CONFIG_DOWN[vid]
    df   = vdict['df'].copy()

    sitting_mask = get_sitting_mask_down(df, vid, cfg['time_offset'])

    df = correct_hips_smooth_down(df, sitting_mask, cfg['alpha_left'], cfg['alpha_right'])
    df = correct_hips_symmetry(df)

    df.to_csv(vdict['out_path'], index=False)
    print(f"down {vid}: alpha_left={cfg['alpha_left']}, alpha_right={cfg['alpha_right']}, time_offset={cfg['time_offset']}, sitting frames={sitting_mask.sum()}")


down 030: alpha_left=-0.4, alpha_right=-0.37, time_offset=0.6, sitting frames=76
down 061: alpha_left=-0.3, alpha_right=-0.25, time_offset=0.8, sitting frames=98
down 076: alpha_left=-0.5, alpha_right=-0.42, time_offset=0.7, sitting frames=91


down 045: alpha_left=-0.35, alpha_right=-0.28, time_offset=0.7, sitting frames=61
down 091: alpha_left=-0.42, alpha_right=-0.38, time_offset=0.3, sitting frames=20
down 093: alpha_left=-0.12, alpha_right=-0.08, time_offset=0.5, sitting frames=81
down 074: alpha_left=-0.35, alpha_right=-0.3, time_offset=0.5, sitting frames=42
down 028: alpha_left=-0.2, alpha_right=-0.12, time_offset=0.5, sitting frames=40


down 047: alpha_left=-0.58, alpha_right=-0.52, time_offset=0.9, sitting frames=106
down 059: alpha_left=-0.1, alpha_right=0, time_offset=0.5, sitting frames=60


## Angle plots

In [6]:
MOLAB_FS = 100
VIDEO_FPS = 50

mapping = [
    ('0028', 'Fp1_SB', '2c'),
    ('0030', 'Fp1_SB', '2e'),
    ('0045', 'Fp2_CF', '2c'),
    ('0047', 'Fp2_CF', '2e'),
    ('0059', 'Fp3_JN', '2c'),
    ('0061', 'Fp3_JN', '2e'),
    ('0074', 'Fp4_AL', '2c'),
    ('0076', 'Fp4_AL', '2e'),
    ('0091', 'Fp5_WL', '2c'),
    ('0093', 'Fp5_WL', '2e'),
]

beep_times = {
    '0028': 9.14,
    '0030': 5.38,
    '0045': 5.88,
    '0047': 6.12,
    '0059': 5.70,
    '0061': 6.10,
    '0074': 5.42,
    '0076': 5.54,
    '0091': 5.52,
    '0093': 5.00,
}

corrections = {
    '0028': 0.52,
}

MOLAB_FULL_DIR = '../data/processed/molab'
ANGLE_PLOTS_DIR = '../data/processed/angle_plots_corrected'
VIDEO_ASPECT = 1920 / 1080

PLOT_JOINTS = [
    ('L_knee_X',  'Left Knee'),
    ('R_knee_X',  'Right Knee'),
    ('L_hip_X',   'Left Hip'),
    ('R_hip_X',   'Right Hip'),
    ('L_ankle_X', 'Left Ankle'),
    ('R_ankle_X', 'Right Ankle'),
    ('pelvis_X',  'Pelvis'),
]

offsets = {}
for vid, fp, mid in mapping:
    full = pd.read_csv(f'{MOLAB_FULL_DIR}/{vid}_molab.csv')
    pre  = full[full['molab_frame'] < 41]
    offsets[vid] = {j: pre[f'JointAngle/{j}'].mean() for j, _ in PLOT_JOINTS}


def joint_angle_2d(ax, ay, bx, by, cx, cy):
    v1x, v1y = ax - bx, ay - by
    v2x, v2y = cx - bx, cy - by
    dot  = v1x * v2x + v1y * v2y
    norm = np.sqrt(v1x**2 + v1y**2) * np.sqrt(v2x**2 + v2y**2) + 1e-8
    return 180 - np.degrees(np.arccos(np.clip(dot / norm, -1, 1)))


def segment_angle_2d(ax, ay, bx, by, cx, cy, dx, dy):
    v1x, v1y = bx - ax, by - ay
    v2x, v2y = dx - cx, dy - cy
    dot  = v1x * v2x + v1y * v2y
    norm = np.sqrt(v1x**2 + v1y**2) * np.sqrt(v2x**2 + v2y**2) + 1e-8
    return 180 - np.degrees(np.arccos(np.clip(dot / norm, -1, 1)))


def trunk_angle_2d(kp):
    mid_hip_x = (kp['left_hip_x'].values + kp['right_hip_x'].values) / 2
    mid_hip_y = (kp['left_hip_y'].values + kp['right_hip_y'].values) / 2
    mid_sh_x  = (kp['left_shoulder_x'].values + kp['right_shoulder_x'].values) / 2
    mid_sh_y  = (kp['left_shoulder_y'].values + kp['right_shoulder_y'].values) / 2
    dx = mid_sh_x - mid_hip_x
    dy = mid_sh_y - mid_hip_y
    return -np.degrees(np.arctan2(dx, -dy))


def compute_kp_angles(kp):
    kp = kp.copy()
    x_cols = [c for c in kp.columns if c.endswith('_x')]
    kp[x_cols] *= VIDEO_ASPECT

    angles = {}
    for side, s in [('L', 'left'), ('R', 'right')]:
        angles[f'{side}_knee_X'] = joint_angle_2d(
            kp[f'{s}_hip_x'].values,    kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,   kp[f'{s}_knee_y'].values,
            kp[f'{s}_ankle_x'].values,  kp[f'{s}_ankle_y'].values,
        )
        angles[f'{side}_hip_X'] = joint_angle_2d(
            kp[f'{s}_shoulder_x'].values, kp[f'{s}_shoulder_y'].values,
            kp[f'{s}_hip_x'].values,      kp[f'{s}_hip_y'].values,
            kp[f'{s}_knee_x'].values,     kp[f'{s}_knee_y'].values,
        )
        angles[f'{side}_ankle_X'] = segment_angle_2d(
            kp[f'{s}_heel_x'].values,       kp[f'{s}_heel_y'].values,
            kp[f'{s}_foot_index_x'].values, kp[f'{s}_foot_index_y'].values,
            kp[f'{s}_ankle_x'].values,      kp[f'{s}_ankle_y'].values,
            kp[f'{s}_knee_x'].values,       kp[f'{s}_knee_y'].values,
        )
    angles['pelvis_X'] = trunk_angle_2d(kp)
    return angles


def plot_molab_angles(molab_dir, kp_dir, kp_cor_dir, kp_pattern, title, out_dir, label, phase_dict, phase_indices):
    os.makedirs(out_dir, exist_ok=True)
    for vid, fp, mid in mapping:
        molab  = pd.read_csv(f'{molab_dir}/{vid}_molab.csv')
        offset = beep_times[vid] - 4 + corrections.get(vid, 0)
        t    = molab['molab_time'].values + offset
        mask = molab['molab_frame'].values >= 41

        kp_files  = glob.glob(f'{kp_dir}/{kp_pattern.format(vid=vid)}')
        kp_angles = None
        kp_time   = None
        if kp_files:
            kp        = pd.read_csv(kp_files[0])
            kp_time   = kp['frame'].values / VIDEO_FPS
            kp_angles = compute_kp_angles(kp)

        kp_cor_files  = glob.glob(f'{kp_cor_dir}/{kp_pattern.format(vid=vid)}')
        kp_cor_angles = None
        kp_cor_time   = None
        if kp_cor_files:
            kp_cor        = pd.read_csv(kp_cor_files[0])
            kp_cor_time   = kp_cor['frame'].values / VIDEO_FPS
            kp_cor_angles = compute_kp_angles(kp_cor)

        vid3 = vid[1:]
        phases = [(phase_dict[vid3][idx], lbl) for idx, lbl in phase_indices if vid3 in phase_dict]

        fig, axes = plt.subplots(4, 2, figsize=(14, 16))
        fig.suptitle(f'{title} {vid}: {fp}', fontsize=13, fontweight='bold')

        for row in range(3):
            axes[row, 1].sharey(axes[row, 0])

        for i, (joint, jlabel) in enumerate(PLOT_JOINTS):
            ax  = axes.flat[i]
            raw = molab[f'JointAngle/{joint}'].values.copy()
            raw[mask] += offsets[vid][joint]
            ax.plot(t, raw, linewidth=1.2, color='steelblue', label='MoLab')

            if kp_angles is not None and joint in kp_angles:
                ax.plot(kp_time, kp_angles[joint], linewidth=1.2,
                        color='tomato', alpha=0.8, label='SAT')

            if kp_cor_angles is not None and joint in kp_cor_angles:
                ax.plot(kp_cor_time, kp_cor_angles[joint], linewidth=1.2,
                        color='green', alpha=0.8, label='SAT corrected')

            for phase_t, phase_lbl in phases:
                ax.axvline(float(phase_t), color='black', linestyle='--', linewidth=0.8)
                ax.text(float(phase_t), 1.0, phase_lbl, transform=ax.get_xaxis_transform(),
                        ha='left', va='bottom', fontsize=8, rotation=20,
                        color='black', clip_on=False)

            ax.set_title(jlabel, pad=28)
            ax.set_xlabel('Video time (s)')
            ax.set_ylabel('Angle (°)')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7)

        axes.flat[-1].set_visible(False)
        plt.tight_layout()
        plt.savefig(f'{out_dir}/{vid}_{fp}_{label}.png', dpi=150, bbox_inches='tight')
        plt.close(fig)


plot_molab_angles(
    '../data/processed/molab_cropped',
    '../data/processed/keypoints_combined',
    '../data/processed/keypoints_corrected',
    'DJI_*_{vid}_D.csv',
    'MoLab vs SAT Sit-to-stand',
    f'{ANGLE_PLOTS_DIR}/sit_to_stand',
    'sit_to_stand',
    phase_dict=UP_TIME_DICT,
    phase_indices=[(1, 'rörelsestart'), (2, 'förberedelsefas'), (4, 'uppresningsfas')],
)
plot_molab_angles(
    '../data/processed/molab_cropped_down',
    '../data/processed/keypoints_combined_down',
    '../data/processed/keypoints_corrected_down',
    'DJI_*_{vid}_D.csv',
    'MoLab vs SAT Stand-to-sit',
    f'{ANGLE_PLOTS_DIR}/stand_to_sit',
    'stand_to_sit',
    phase_dict=DOWN_TIME_DICT,
    phase_indices=[(1, 'rörelsestart'), (2, 'nedåtgående'), (3, 'bakåtgående')],
)
print('Angle plots done.')

Angle plots done.


## Save videos

* Blue = corrected
* Green = original

In [7]:
import imageio
import cv2
import re

VIDEO_DIR       = Path('../data/original/Utvalda filminspelningar för IRAF analys/Dec 2025 sit-stå och stå-sitt')

OUT_UP_DIR = Path('../data/processed/keypoints_corrected')
OUT_DOWN_DIR = Path('../data/processed/keypoints_corrected_down')

OUT_UP_ORIG_DIR = Path('../data/processed/keypoints_combined')
OUT_DOWN_ORIG_DIR = Path('../data/processed/keypoints_combined_down')

OUT_UP_VIDEOS = Path('../data/processed/keypoints_corrected_videos')
OUT_DOWN_VIDEOS = Path('../data/processed/keypoints_corrected_down_videos')

time_dict = {
    '028': (9.90,  11.77), '030': (5.80,  7.63),
    '045': (6.54,   7.98), '047': (7.03,  8.91),
    '059': (6.65,   9.11), '061': (6.00,  9.00),
    '074': (6.07,   7.27), '076': (5.00,  8.00),
    '091': (6.39,   7.61), '093': (5.86,  7.67),
}

stand_to_sit_time_dict = {
    '028': (11.87, 12.87), '030': (8.07,  11.56),
    '045': (7.94,   9.26), '047': (9.20,  12.30),
    '059': (9.11,  11.06), '061': (10.57, 13.41),
    '074': (7.27,   8.20), '076': (8.27,  10.84),
    '091': (7.56,   8.18), '093': (8.50,  11.19),
}

JOINTS = [
    'nose', 'left_ear',
    'left_shoulder', 'right_shoulder',
    'left_elbow',    'right_elbow',
    'left_wrist',    'right_wrist',
    'left_hip',      'right_hip',
    'left_knee',     'right_knee',
    'left_ankle',    'right_ankle',
    'left_heel',     'right_heel',
    'left_foot_index', 'right_foot_index',
]

CORRECTION_TOL = 1e-6
COLOR_ORIG = (80, 200, 80)
COLOR_CORR = (80, 80, 255)


def draw_keypoints_cv2(image, orig_row, corr_row):
    img = image.copy()
    height, width = img.shape[:2]
    for joint in JOINTS:
        ox = orig_row.get(f'{joint}_x')
        oy = orig_row.get(f'{joint}_y')
        cx = corr_row.get(f'{joint}_x')
        cy = corr_row.get(f'{joint}_y')
        if ox is None or oy is None or (isinstance(ox, float) and np.isnan(ox)):
            continue
        px, py = int(ox * width), int(oy * height)
        cv2.circle(img, (px, py), 6, COLOR_ORIG, -1)
        corrected = (cx is not None and cy is not None and
                     not (isinstance(cx, float) and np.isnan(cx)) and
                     (abs(cx - ox) > CORRECTION_TOL or abs(cy - oy) > CORRECTION_TOL))
        if corrected:
            qx, qy = int(cx * width), int(cy * height)
            cv2.circle(img, (qx, qy), 6, COLOR_CORR, -1)
    return img



def get_video_rotation(video_path):
    import subprocess, json as _json
    try:
        r = subprocess.run(
            ['ffprobe', '-v', 'quiet', '-print_format', 'json', '-show_streams', str(video_path)],
            capture_output=True, text=True
        )
        d = _json.loads(r.stdout)
        for s in d.get('streams', []):
            if s.get('codec_type') == 'video':
                for sd in s.get('side_data_list', []):
                    rot = sd.get('rotation', 0)
                    if rot:
                        return int(rot)
    except Exception:
        pass
    return 0


for corr_dir, orig_dir, out_video, td, label in [
    (OUT_UP_DIR,   OUT_UP_ORIG_DIR,   OUT_UP_VIDEOS,   time_dict,              'sit-to-stand'),
    (OUT_DOWN_DIR, OUT_DOWN_ORIG_DIR, OUT_DOWN_VIDEOS, stand_to_sit_time_dict, 'stand-to-sit'),
]:
    out_video.mkdir(parents=True, exist_ok=True)
    print(f'\n=== {label} ===')
    for csv_path in sorted(corr_dir.glob('*.csv')):
        match = re.search(r'_0(\d{3})_D', csv_path.stem)
        if not match:
            continue
        vid_id    = match.group(1)
        vid_files = list(VIDEO_DIR.glob(f'*_0{vid_id}_D.MP4'))

        if not vid_files:
            print(f'  Video not found: {vid_id}')
            continue

        orig_path = orig_dir / csv_path.name
        if not orig_path.exists():
            print(f'  Original not found: {orig_path.name}')
            continue

        corr_df = pd.read_csv(csv_path)
        orig_df = pd.read_csv(orig_path)
        out_path    = out_video / f'{csv_path.stem}.mp4'
        start_frame = int(td[vid_id][0] * VIDEO_FPS)

        cap = cv2.VideoCapture(str(vid_files[0]))
        vid_rotation = get_video_rotation(vid_files[0])
        cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

        with imageio.get_writer(str(out_path), fps=VIDEO_FPS, format='ffmpeg') as writer:
            for corr_row, orig_row in zip(corr_df.itertuples(), orig_df.itertuples()):
                ret, frame_bgr = cap.read()
                if not ret:
                    break
                if vid_rotation == -180 or vid_rotation == 180:
                    frame_bgr = cv2.rotate(frame_bgr, cv2.ROTATE_180)
                frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                annotated = draw_keypoints_cv2(frame_rgb, orig_row._asdict(), corr_row._asdict())
                writer.append_data(annotated)

        cap.release()
        print(f'  {vid_id}: {len(corr_df)} frames -> {out_path.name}')

print('\nDone.')



=== sit-to-stand ===


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  028: 94 frames -> DJI_20250425092743_0028_D.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  030: 92 frames -> DJI_20250425093100_0030_D.mp4


...

  076: 129 frames -> DJI_20250425121226_0076_D.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  091: 32 frames -> DJI_20250425125202_0091_D.mp4


IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (1920, 1080) to (1920, 1088) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).


  093: 135 frames -> DJI_20250425125448_0093_D.mp4

Done.
